In [ ]:
%run scripts/verify_environment.py

verify_environment()

In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
idp_name_2 = 'FakeCAS'
idp_username_2 = None
idp_password_2 = None
weko_url = 'https://weko3.rdm.example.com/'
weko_admin_email = None
weko_admin_password = None
weko_user_email = None
weko_user_password = None
weko_index_name = 'GRDM-Collaboration-Test-VR-1'
default_result_path = None
close_on_fail = False
transition_timeout = 60000
browser_type = 'chromium'
ignore_https_errors = False
project_name = 'TEST-WEKO-202512221343'
project_url = None
default_storage_label = 'NII Storage'
use_files_tab = False
oauth_client_type = 'direct'

In [ ]:
if idp_username_2 is None:
    prompt = f"Username for {idp_name_2 or 'user account'}: "
    idp_username_2 = input(prompt)
if idp_password_2 is None:
    prompt = f"Password for {idp_username_2}: "
    idp_password_2 = getpass(prompt)

if weko_user_email is None:
    weko_user_email = input('WEKO user email: ')
if weko_user_password is None:
    weko_user_password = getpass('WEKO user password: ')
if weko_admin_email is None:
    weko_admin_email = input('WEKO admin email: ')
if weko_admin_password is None:
    weko_admin_password = getpass('WEKO admin password: ')

if project_name is None:
    project_name = datetime.now().strftime('TEST-WEKO-%Y%m%d%H%M')

# メタデータのWEKOへの根拠データ送信

- サブシステム名: アドオン
- ページ/アドオン: Metadata, WEKO
- 機能分類: メタデータの送信
- シナリオ名: メタデータのWEKOへの送信
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM一般ユーザー, WEKO教員以上)
- 事前条件: 「プロジェクトに対するWEKOアドオンの登録」を実施済みであること

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import yaml
import json
import importlib
import os

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm
from scripts.metadata_v2025 import ProjectMetadataForm, FileMetadataForm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

In [ ]:
from pathlib import Path
from docx import Document

root_evidence_doc_path = Path(work_dir) / 'evidence_notes.docx'
root_evidence_doc = Document()
root_evidence_doc.add_heading('Evidence data package', level=1)
root_evidence_doc.add_paragraph('Auto-generated file for WEKO metadata registration tests.')
root_evidence_doc.save(root_evidence_doc_path)

root_evidence_summary_path = Path(work_dir) / 'evidence_overview.txt'
root_evidence_summary_path.write_text('Summary of evidence data prepared for WEKO metadata registration tests.\n', encoding='utf-8')

root_evidence_file_paths = [str(root_evidence_doc_path), str(root_evidence_summary_path)]


In [ ]:
project_url = None

async def goto_file_view(page):
    url = project_url
    if use_files_tab:
        if not url.endswith('/'):
            url += '/'
        url += 'files/'
    await page.goto(url)

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()
    await grdm.expect_anonymous_toppage(page, idp_name_2, transition_timeout=transition_timeout)

await run_pw(_step, new_page=True)

## IdPを利用し、既存ユーザー1としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

作成したプロジェクトのプロジェクトダッシュボードが表示されること

In [ ]:
project_page = None

async def _step(page):
    project_link = page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]')
    await expect(project_link).to_be_visible(timeout=transition_timeout)
    await project_link.click()
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    global project_url, project_page
    project_url = page.url
    project_page = page

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メタデータ」をクリックする

メタデータの一覧ページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "メタデータ")]').click()
    await expect(page.locator('//button[@data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「新規メタデータ」をクリックする

スキーマ選択ダイアログが表示されること

In [ ]:
import asyncio

async def _step(page):
    await page.locator('//button[@data-test-new-metadata-button]').click()
    await expect(page.locator('//*[@data-test-new-report-modal-schema="公的資金による研究データのメタデータ登録"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//button[text()="メタデータを作成"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「メタデータを作成」をクリックする

メタデータ編集ウィンドウが表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[text() = "メタデータを作成"]').click()
    await expect(page.locator('//*[contains(text(), "資金配分機関情報")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 全ての入力欄に正常系の値を入力する

正常系の値は以下の通り:
- 資金配分機関情報: 国立研究開発法人日本医療研究開発機構 | AMED
- 体系的番号におけるプログラム情報コード: abc12
- プログラム名 (日本語): プログラムA
- Program name (English): ProgramA
- 体系的番号: TEST123456
- プロジェクト名 (日本語): プロジェクトA
- Project name (English): ProjectA
- プロジェクトの分野: 社会基盤 | 789

各フィールドに入力した値が表示されること

In [ ]:
importlib.reload(scripts.metadata_v2025)
from scripts.metadata_v2025 import ProjectMetadataForm, FileMetadataForm

async def _step(page):
    form = ProjectMetadataForm(page)
    
    await form.fill("資金配分機関情報", "国立研究開発法人日本医療研究開発機構 | AMED")
    await form.fill("体系的番号におけるプログラム情報コード", "abc12")
    await form.fill("プログラム名 (日本語)", "プログラムA")
    await form.fill("Program name (English)", "ProgramA")
    await form.fill("体系的番号", "TEST123456")
    await form.fill("プロジェクト名 (日本語)", "プロジェクトA")
    await form.fill("Project name (English)", "ProjectA")
    await form.fill("プロジェクトの分野", "社会基盤 | 789")
    
    # Verify values
    assert "AMED" in await form.get("資金配分機関情報")
    assert await form.get("体系的番号におけるプログラム情報コード") == "abc12"
    assert await form.get("プログラム名 (日本語)") == "プログラムA"
    assert await form.get("Program name (English)") == "ProgramA"
    assert "TEST123456" in await form.get("体系的番号")
    assert await form.get("プロジェクト名 (日本語)") == "プロジェクトA"
    assert await form.get("Project name (English)") == "ProjectA"
    assert "789" in await form.get("プロジェクトの分野")

await run_pw(_step)

## 「次へ」をクリックする

左側ペイン「助成情報登録」が緑色かつチェックマークが表示されること

In [ ]:
import re

async def _step(page):
    await page.locator('//a[@data-test-goto-next-page]').click()
    await expect(page.locator('//span[@data-test-label and text() = "助成情報登録"]/../preceding-sibling::i')).to_have_class(re.compile(r'.*fa-check-circle-o.*'), timeout=transition_timeout)
    await asyncio.sleep(10)

await run_pw(_step)

## プロジェクト名「TEST-WEKO-YYYYMMDDHHMM」をクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-analytics-name="Go to project"]').click()
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メタデータ」をクリックする

メタデータの一覧ページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "メタデータ")]').click()
    await expect(page.locator('//*[@data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「新規メタデータ」をクリックし「メタデータを作成」をクリックする

メタデータ編集ウィンドウが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-new-metadata-button]').click()
    await expect(page.locator('//*[@data-test-new-report-modal-create-report-button]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[@data-test-new-report-modal-schema="公的資金による研究データのメタデータ登録"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//*[@data-test-new-report-modal-create-report-button]').click()
    await expect(page.locator('//*[contains(text(), "資金配分機関情報")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 必須入力欄に正常系の値を入力する

正常系の値は以下の通り:
- 資金配分機関情報: 国立研究開発法人日本医療研究開発機構 | AMED
- 体系的番号: TEST123456
- プロジェクト名 (日本語): プロジェクトA必須
- プロジェクトの分野: 社会基盤 | 789

各フィールドに入力した値が表示されること

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    
    await form.fill("資金配分機関情報", "国立研究開発法人日本医療研究開発機構 | AMED")
    await form.fill("体系的番号", "TEST123456")
    await form.fill("プロジェクト名 (日本語)", "プロジェクトA必須")
    await form.fill("プロジェクトの分野", "社会基盤 | 789")
    
    # Verify values
    assert "AMED" in await form.get("資金配分機関情報")
    assert "TEST123456" in await form.get("体系的番号")
    assert await form.get("プロジェクト名 (日本語)") == "プロジェクトA必須"
    assert "789" in await form.get("プロジェクトの分野")

await run_pw(_step)

## 「次へ」をクリックする

左側ペイン「助成情報登録」が緑色かつチェックマークが表示されること

In [ ]:
import re

async def _step(page):
    await page.locator('//a[@data-test-goto-next-page]').click()
    await expect(page.locator('//span[@data-test-label and text() = "助成情報登録"]/../preceding-sibling::i')).to_have_class(re.compile(r'.*fa-check-circle-o.*'), timeout=transition_timeout)

await run_pw(_step)

## プロジェクト名「TEST-WEKO-YYYYMMDDHHMM」をクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    await asyncio.sleep(10)
    await page.locator('//*[@data-analytics-name="Go to project"]').click()
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## メタデータ一覧の「下書き」をクリックする

以下のメタデータが表示されること
- プロジェクトA必須
- プロジェクトA

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "メタデータ")]').click()
    await expect(page.locator('//*[@data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[text() = "プロジェクトA必須"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[text() = "プロジェクトA"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードまたはファイルタブの「JAIRO Cloud: (インデックス名)」をクリックする

「新規フォルダ作成」ボタンが表示されること (表示まで数秒かかる可能性がある)

> プロジェクトダッシュボードまたはファイルタブのどちらを使うかは、テストシートにより決定する

In [ ]:
async def _step(page):
    await page.locator(f'//a[contains(@class, "project-title") and text() = "{project_name}"]').click()
    if use_files_tab:
        await page.locator('//*[@id = "projectNavFiles"]//a[contains(text(), "ファイル")]').click()
    await expect(grdm.get_select_storage_title_locator(page, default_storage_label)).to_be_visible(timeout=transition_timeout)
    target_label = f'JAIRO Cloud: {weko_index_name}'
    await grdm.get_select_storage_title_locator(page, target_label).click()
    await expect(page.locator('//*[text() = "新規フォルダ作成"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## サンプル.pngファイルをアップロードする

テスト環境の「サンプル.png」をJAIRO Cloud配下にドラッグ＆ドロップする。

In [ ]:
import os

sample_file_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'testdata', 'サンプル.png')

async def _step(page):
    target_label = f'JAIRO Cloud: {weko_index_name}'
    dropzone = grdm.get_select_storage_title_xpath(target_label)
    await grdm.drop_file(page, dropzone, sample_file_path)
    await expect(grdm.get_select_file_title_locator(page, 'サンプル.png')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「サンプル.png」のアイコンをクリックする

「メタデータ編集」ボタンが表示されること (表示まで数秒かかる可能性がある)

In [ ]:
async def _step(page):
    await page.locator('//span[text() = "サンプル.png"]/../../..//div[contains(@class, "file-extension")]').click()
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「メタデータ編集」をクリックする

「メタデータ編集」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()
    await expect(page.locator('//h3[contains(text(), "ファイルメタデータの編集")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「メタデータ様式」に「研究成果（公的資金対応）のメタデータ登録」を入力する

「データNo.」フィールドが表示されること

In [ ]:
async def _step(page):
    schema_select = page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')
    await schema_select.select_option(label="研究成果（公的資金対応）のメタデータ登録")
    await expect(page.locator('//label[contains(text(), "データ No.")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 全ての入力欄に正常系の値を入力する

正常系の値は以下の通り:
- データ No.: 123
- データの名称 (日本語): サンプルデータ
- Title (English): sample data
- 掲載日・掲載更新日: 2030-01-01
- データの説明 (日本語): データの説明
- Description (English): data description
- データの分野: ライフサイエンス
- データ種別: 集計データ
- 概略データ量: 1KB
- 管理対象データの利活用・提供方針 (有償/無償): 無償
- 管理対象データの利活用・提供方針 (ライセンス): CC0 1.0 Universal
- 管理対象データの利活用・提供方針 (引用方法等・日本語): 引用方法の説明
- Data utilization and provision policy (citation information, English): citation information
- アクセス権: 公開期間猶予
- 公開予定日: 2030-01-01
- リポジトリ情報 (日本語): リポジトリの情報
- Repository information (English): repository information
- リポジトリURL・DOIリンク: https://example.com/example
- データ作成者: [{e-Rad: '123', 日本語: '情報データ太郎', English: 'Taro Data Joho'}, {e-Rad: '456', 日本語: 'あああ', English: 'aaa'}]
- データ管理機関 (日本語): データ管理機関
- Hosting institution (English): Hosting institution
- データ管理機関コード: code
- データ管理者の種類: 個人
- データ管理者の e-Rad 研究者番号: 123
- データ管理者 (日本語): 情報データ太郎
- Data manager (English): Taro Data Joho
- データ管理者の所属組織名 (日本語): データ管理者の所属組織名
- Contact organization of data manager (English): Contact organization of data manager
- データ管理者の所属機関の連絡先住所 (日本語): 愛知県 名古屋市
- Contact address of data manager (English): Aichi, Nagoya
- データ管理者の所属機関の連絡先電話番号: 123-456-789
- データ管理者の所属機関の連絡先メールアドレス: sample@example.com
- 備考 (日本語): 備考123
- Remarks (English): Remarks123

各フィールドに入力した値が表示されること

In [ ]:
importlib.reload(scripts.metadata_v2025)
from scripts.metadata_v2025 import ProjectMetadataForm, FileMetadataForm

async def _step(page):
    form = FileMetadataForm(page)
    
    await form.fill("データ No.", "123")
    await form.fill("データの名称 (日本語)", "サンプルデータ")
    await form.fill("Data title (English)", "sample data")
    await form.fill("掲載日・掲載更新日", "2030-01-01")
    await form.fill("データの説明 (日本語)", "データの説明")
    await form.fill("Description (English)", "data description")
    await form.fill("データの分野", "ライフサイエンス")
    await form.fill("データ種別", "集計データ")
    await form.fill("概略データ量", "1KB")
    await form.fill("管理対象データの利活用・提供方針 (有償/無償)", "無償")
    await form.fill("管理対象データの利活用・提供方針 (ライセンス)", "CC0 1.0 Universal")
    await form.fill("管理対象データの利活用・提供方針 (引用方法等・日本語)", "引用方法の説明")
    await form.fill("Data utilization and provision policy (citation information, English)", "citation information")
    await form.fill("アクセス権", "公開期間猶予")
    await form.fill("公開予定日", "2030-01-01")
    await form.fill("リポジトリ情報 (日本語)", "リポジトリの情報")
    await form.fill("Repository information (English)", "repository information")
    await form.fill("リポジトリURL・DOIリンク", "https://example.com/example")
    
    # Table: データ作成者
    await form.fill_author({
        'number': '123',
        'name_ja': {'last': '情報', 'middle': 'データ', 'first': '太郎'},
        'name_en': {'last': 'Joho', 'middle': 'Data', 'first': 'Taro'},
        'affiliation_ja': '',
        'affiliation_en': '',
    }, table_label='データ作成者', add_row=False)
    await form.fill_author({
        'number': '456',
        'name_ja': {'last': 'あ', 'middle': 'あ', 'first': 'あ'},
        'name_en': {'last': 'a', 'middle': 'a', 'first': 'a'},
        'affiliation_ja': '',
        'affiliation_en': '',
    }, table_label='データ作成者')
    
    await form.scroll_to("データ管理機関 (日本語)")
    await form.fill("データ管理機関 (日本語)", "データ管理機関")
    await form.fill("Hosting institution (English)", "Hosting institution")
    await form.fill("データ管理機関コード", "code")
    await form.fill("データ管理者の種類", "個人")
    await form.fill("データ管理者の e-Rad 研究者番号", "123")
    await form.fill_name("データ管理者 (日本語)", {'last': '情報', 'middle': 'データ', 'first': '太郎'})
    await form.fill_name("Data manager (English)", {"last": "Joho", "middle": "Data", "first": "Taro"})
    await form.fill("データ管理者の所属組織名 (日本語)", "データ管理者の所属組織名")
    await form.fill("Contact organization of data manager (English)", "Contact organization of data manager")
    await form.fill("データ管理者の所属機関の連絡先住所 (日本語)", "愛知県 名古屋市")
    await form.fill("Contact address of data manager (English)", "Aichi, Nagoya")
    await form.fill("データ管理者の所属機関の連絡先電話番号", "123-456-789")
    await form.fill("データ管理者の所属機関の連絡先メールアドレス", "sample@example.com")
    await form.fill("備考 (日本語)", "備考123")
    await form.fill("Remarks (English)", "Remarks123")
    
    # Verify some key values
    await form.scroll_to("データ No.")
    assert await form.get("データ No.") == "123"
    assert await form.get("データの名称 (日本語)") == "サンプルデータ"

await run_pw(_step)

## WEKOのURLを開く

「Log in」ボタンが表示されること

In [ ]:
async def _step(page):
    await page.goto(weko_url)
    await expect(page.locator('//a[contains(@class, "login-button")]')).to_be_visible(timeout=transition_timeout)

    await asyncio.sleep(5)

await run_pw(_step, new_page=True)

## 「ログイン」をクリックする

WEKO3のログイン画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(@class, "login-button")]').click()
    await expect(page.locator('//input[@name = "email"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## WEKOアカウントを入力し、「Log in」をクリックする

WEKO3のアイテム画面が表示されること


In [ ]:
async def _step(page):
    await page.locator('//input[@name = "email"]').fill(weko_admin_email)
    await page.locator('//input[@name = "password"]').fill(weko_admin_password)
    await page.locator('//button[@type = "submit"]').click()
    await expect(page.locator('//a[text() = "WorkFlow"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)

await run_pw(_step)


## WorkFlowタブをクリックする

WorkFlow画面が表示されること

In [ ]:
current_activities = None
weko_workflow_page = None

async def _step(page):
    global current_activities, weko_workflow_page
    await page.locator('//a[text() = "WorkFlow"]').click()
    await expect(page.locator('#myTabContent').locator('.table-responsive').locator('//th[text() = "Activity"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    current_activities = await page.locator('.activity-link').count()
    weko_workflow_page = page

await run_pw(_step)

## GakuNin RDMのページに戻り、「保存してJAIRO Cloudに登録」をクリックする

ファイルメタデータの編集ダイアログが非表示になり、ファイルの登録ダイアログが表示されること

In [ ]:
async def _step(page):
    page = project_page
    await page.locator('//a[text()="保存してJAIRO Cloudに登録"]').click()
    await expect(page.locator('//h3[text() = "ファイルの登録"]')).to_be_visible(timeout=transition_timeout)
    return page

await run_pw(_step)

## 「プロジェクトA」のチェックボックスをクリックする

チェックボックスがチェックされた状態になること

In [ ]:
async def _step(page):
    checkbox = page.locator('//div[@class="modal fade in"]//*[contains(text(), "プロジェクトA (")]/..//input[@type = "checkbox"]')
    await checkbox.click()
    await expect(checkbox).to_be_checked(timeout=transition_timeout)

await run_pw(_step)

## 「OK」をクリックする

「登録結果」ダイアログが表示されること

In [ ]:
def get_result_dialog_locator(page):
    if oauth_client_type == 'workflow':
        return page.locator('//*[contains(text(), "審査をお待ちください")]')
    return page.locator('//*[contains(text(), "登録が完了しました")]')

async def _step(page):
    await page.locator('//div[@class="modal fade in"]//a[text()="OK"]').click()
    await expect(get_result_dialog_locator(page)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)

await run_pw(_step)

## 「OK」をクリックする

ダイアログが閉じること。Directモードの場合は、「JAIRO Cloud」配下に「サンプルデータ」アイテムが作成されること

In [ ]:
async def _step(page):
    await get_result_dialog_locator(page).locator('../../..//a[text()="OK"]').click()
    if oauth_client_type != 'workflow':
        await expect(page.locator('//*[contains(@class, "weko-row")]//*[text() = "サンプルデータ"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)

await run_pw(_step)

## WEKOの画面に戻り、リロードする

workflowの場合、Activityが増えること

In [ ]:
async def _step(page):
    page = weko_workflow_page
    await page.reload()
    await expect(page.locator('#myTabContent').locator('.table-responsive').locator('//th[text() = "Activity"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    expected_activities = current_activities
    if oauth_client_type == 'workflow':
        expected_activities += 1
    await expect(page.locator('.activity-link')).to_have_count(expected_activities, timeout=transition_timeout)
    return page

await run_pw(_step)


## (Workflowの場合)一番上のActivityの「表示」をクリックする

「Approval」ボタンが有効になること

In [ ]:
async def _step(page):
    if oauth_client_type != 'workflow':
        return
    added_activity_link = page.locator('.activity-link').first
    await added_activity_link.click()

    await expect(page.locator('//button[text() = "Approval"]')).to_be_enabled(timeout=transition_timeout)
    await asyncio.sleep(5)

await run_pw(_step)

## (Workflowの場合)「Approval」ボタンをクリックする

「Access」ボタンが有効になること

In [ ]:
async def _step(page):
    if oauth_client_type != 'workflow':
        return
    await page.locator('//button[text() = "Approval"]').click()

    await expect(page.locator('//button[contains(text(), "Access")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)

await run_pw(_step)

## (Workflowの場合)「Access」ボタンをクリックする

アイテム詳細画面が表示されること

In [ ]:
async def _step(page):
    if oauth_client_type != 'workflow':
        return
    await page.locator('//button[contains(text(), "Access")]').click()

    await expect(page.locator('//a[contains(text(), "JSON")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#public_status_form')).to_contain_text('Public', timeout=transition_timeout)
    await asyncio.sleep(5)

await run_pw(_step)

## (Directの場合)「サンプルデータ」をクリックする

新たにタブが開き、「JSON」ボタンが表示されること。また、Publicであること

In [ ]:
async def _step(page):
    if oauth_client_type == 'workflow':
        return
    page= project_page
    popup_future = page.context.wait_for_event('page')
    await page.locator('//*[contains(@class, "weko-row")]//*[text() = "サンプルデータ"]').click()
    popup = await popup_future
    await popup.wait_for_load_state()
    await expect(popup.locator('//a[contains(text(), "JSON")]')).to_be_visible(timeout=transition_timeout)
    await expect(popup.locator('#public_status_form')).to_contain_text('Public', timeout=transition_timeout)
    return popup

await run_pw(_step)

## WEKOのJSON出力を確認する（全項目入力）

JSON出力に含まれるmetadataが以下の内容と一致すること

In [ ]:
import difflib

async def _step(page):
    data_expected = """
      item_30002_title0:
        attribute_name: Title
        attribute_value_mlt:
        - subitem_title: sample data
          subitem_title_language: en
        - subitem_title: サンプルデータ
          subitem_title_language: ja
      item_30002_creator2:
        attribute_name: Creator
        attribute_type: creator
        attribute_value_mlt:
        - creatorNames:
          - creatorName: "Joho, Data Taro"
            creatorNameLang: en
          - creatorName: "情報, データ 太郎"
            creatorNameLang: ja
          familyNames:
          - familyName: Joho
            familyNameLang: en
          - familyName: 情報
            familyNameLang: ja
          givenNames:
          - givenName: Data Taro
            givenNameLang: en
          - givenName: データ 太郎
            givenNameLang: ja
          nameIdentifiers:
          - nameIdentifier: '123'
            nameIdentifierScheme: e-Rad_Researcher
        - creatorNames:
          - creatorName: "a, a a"
            creatorNameLang: en
          - creatorName: "あ, あ あ"
            creatorNameLang: ja
          familyNames:
          - familyName: a
            familyNameLang: en
          - familyName: あ
            familyNameLang: ja
          givenNames:
          - givenName: a a
            givenNameLang: en
          - givenName: あ あ
            givenNameLang: ja
          nameIdentifiers:
          - nameIdentifier: '456'
            nameIdentifierScheme: e-Rad_Researcher
      item_30002_access_rights4:
        attribute_name: Access Rights
        attribute_value_mlt:
        - subitem_access_right: embargoed access
          subitem_access_right_uri: "http://purl.org/coar/access_right/c_f1cf"
      item_30002_rights6:
        attribute_name: Rights
        attribute_value_mlt:
        - subitem_rights: citation information
          subitem_rights_language: en
        - subitem_rights: 引用方法の説明
          subitem_rights_language: ja
        - subitem_rights: free
          subitem_rights_language: en
        - subitem_rights: 無償
          subitem_rights_language: ja
        - subitem_rights: CC0 1.0 Universal
          subitem_rights_language: en
          subitem_rights_resource: https://creativecommons.org/publicdomain/zero/1.0/deed.en
        - subitem_rights: CC0 1.0 Universal
          subitem_rights_language: ja
          subitem_rights_resource: https://creativecommons.org/publicdomain/zero/1.0/deed.en
      item_30002_subject8:
        attribute_name: Subject
        attribute_value_mlt:
        - subitem_subject: Life Science
          subitem_subject_language: en
          subitem_subject_scheme: e-Rad_field
        - subitem_subject: ライフサイエンス
          subitem_subject_language: ja
          subitem_subject_scheme: e-Rad_field
      item_30002_description9:
        attribute_name: Description
        attribute_value_mlt:
        - subitem_description: data description
          subitem_description_language: en
          subitem_description_type: Abstract
        - subitem_description: データの説明
          subitem_description_language: ja
          subitem_description_type: Abstract
      # TODO: item_30002_date11 is not present in actual metadata - needs investigation
      item_30002_funding_reference21:
        attribute_name: Funding Reference
        attribute_value_mlt:
        - subitem_award_numbers:
            subitem_award_number: TEST123456
            subitem_award_number_type: JGN
          subitem_award_titles:
          - subitem_award_title: ProjectA
            subitem_award_title_language: en
          - subitem_award_title: プロジェクトA
            subitem_award_title_language: ja
          subitem_funder_identifiers:
            subitem_funder_identifier_type: ROR
            subitem_funder_identifier: "https://ror.org/004rtk039"
          subitem_funder_names:
          - subitem_funder_name: Japan Agency for Medical Research and Development(AMED)
            subitem_funder_name_language: en
          - subitem_funder_name: 国立研究開発法人日本医療研究開発機構(AMED)
            subitem_funder_name_language: ja
          subitem_funding_stream_identifiers:
            subitem_funding_stream_identifier: abc12
            subitem_funding_stream_identifier_type: JGN_fundingStream
          subitem_funding_streams:
          - subitem_funding_stream: ProgramA
            subitem_funding_stream_language: en
          - subitem_funding_stream: プログラムA
            subitem_funding_stream_language: ja
      item_30002_resource_type13:
        attribute_name: Resource Type
        attribute_value_mlt:
        - resourcetype: aggregated data
          resourceuri: http://purl.org/coar/resource_type/ACF7-8YT9
      item_30002_contributor3:
        attribute_name: Contributor
        attribute_value_mlt:
        - contributorNames:
          - contributorName: 'Contact organization of data manager Aichi, Nagoya TEL:
              123-456-789 E-Mail: sample@example.com'
            lang: en
            nameType: Organizational
          - contributorName: 'データ管理者の所属組織名 愛知県 名古屋市 TEL: 123-456-789 E-Mail: sample@example.com'
            lang: ja
            nameType: Organizational
          contributorType: ContactPerson
        - contributorNames:
          - contributorName: "Joho, Data Taro"
            lang: en
          - contributorName: "情報, データ 太郎"
            lang: ja
          contributorType: DataManager
          familyNames:
          - familyName: Joho
            familyNameLang: en
          - familyName: 情報
            familyNameLang: ja
          givenNames:
          - givenName: Data Taro
            givenNameLang: en
          - givenName: データ 太郎
            givenNameLang: ja
          nameIdentifiers:
          - nameIdentifierScheme: e-Rad_Researcher
            nameIdentifier: '123'
          contributorAffiliations:
          - contributorAffiliationNames:
            - contributorAffiliationName: Contact organization of data manager
              contributorAffiliationNameLang: en
            - contributorAffiliationName: データ管理者の所属組織名
              contributorAffiliationNameLang: ja
        - contributorNames:
          - contributorName: Hosting institution
            lang: en
            nameType: Organizational
          - contributorName: データ管理機関
            lang: ja
            nameType: Organizational
          contributorType: HostingInstitution
          nameIdentifiers:
          - nameIdentifierScheme: ROR
            nameIdentifier: code

    """
    await page.locator('//a[contains(text(), "JSON")]').click()
    await expect(page.locator('//pre')).to_be_visible(timeout=transition_timeout)
    actual_metadata = json.loads(await page.locator('//pre').inner_text())['metadata']
    expected_metadata = yaml.safe_load(data_expected)
    for key, expected in expected_metadata.items():
        message = f'Missing key: {key} in actual metadata \nActual metadata: {json.dumps(actual_metadata, ensure_ascii=False, indent=2, sort_keys=True)}'
        assert key in actual_metadata, message
        actual = actual_metadata[key]
        assert key in actual_metadata, f'Missing key: {key}'
        actual_str = json.dumps(actual, ensure_ascii=False, indent=2, sort_keys=True)
        expected_str = json.dumps(expected, ensure_ascii=False, indent=2, sort_keys=True)
        diff = '\n'.join(
            difflib.unified_diff(
                expected_str.splitlines(),
                actual_str.splitlines(),
                fromfile='expected',
                tofile='actual',
                lineterm=''
            )
        )
        message = (
            f"Metadata mismatch for '{key}'.\n"
            f"{diff or 'No textual diff (objects differ structurally).'}"
        )
        assert actual == expected, message

await run_pw(_step)

## WEKOで「Delete」をクリックし、アイテムを削除する

アイテムが削除されること

In [ ]:
async def _step(page):
    await page.go_back()
    await page.locator('//a[@id = "btn_delete"]').click()
    await expect(page.locator('//button[@id = "confirm_delete_button"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[@id = "confirm_delete_button"]').click()
    await expect(page.locator('//*[contains(text(), "This item has been deleted")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(transition_timeout / 1000 / 2)

await run_pw(_step)

## プロジェクトダッシュボードに戻り、「JAIRO Cloud」配下にファイルをアップロードする（必須項目のみ入力テスト）

ファイルがアップロードされること

In [ ]:
await close_latest_page()

async def _step(page):
    await goto_file_view(page)
    await expect(grdm.get_select_storage_title_locator(page, default_storage_label)).to_be_visible(timeout=transition_timeout)
    target_label = f'JAIRO Cloud: {weko_index_name}'
    await grdm.get_select_storage_title_locator(page, target_label).click()
    await expect(page.locator('//*[text() = "新規フォルダ作成"]')).to_be_visible(timeout=transition_timeout)
    
    dropzone = grdm.get_select_storage_title_xpath(target_label)
    await grdm.drop_file(page, dropzone, sample_file_path)
    await expect(grdm.get_select_file_title_locator(page, 'サンプル.png')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「サンプル.png」を開く

ファイルの詳細画面が表示されること

In [ ]:
async def _step(page):
    await grdm.get_select_file_title_locator(page, 'サンプル.png').click(timeout=transition_timeout)
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    global project_page
    project_page = page

await run_pw(_step)

## ファイルのメタデータ編集ダイアログを開き、必須項目のみ入力する

必須項目:
- データの名称 (日本語): サンプルデータ
- データの説明 (日本語): データの説明
- データの分野: ライフサイエンス
- データ種別: 集計データ
- 管理対象データの利活用・提供方針 (有償/無償): 無償
- 管理対象データの利活用・提供方針 (ライセンス): CC0 1.0 Universal
- 管理対象データの利活用・提供方針 (引用方法等・日本語): 引用方法の説明
- Data utilization and provision policy (citation information, English): citation information
- アクセス権: 公開
- データ管理機関 (日本語): データ管理機関
- データ管理者の種類: 個人
- データ管理者 (日本語): 情報データ太郎


In [ ]:
async def _step(page):
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//*[text() = "メタデータ編集"]').click()
    await expect(page.locator('//h3[contains(text(), "ファイルメタデータの編集")]')).to_be_visible(timeout=transition_timeout)
    
    schema_select = page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')
    await schema_select.select_option(label="研究成果（公的資金対応）のメタデータ登録")
    await expect(page.locator('//label[contains(text(), "データ No.")]')).to_be_visible(timeout=transition_timeout)
    
    form = FileMetadataForm(page)
    
    await form.fill("データの名称 (日本語)", "サンプルデータ")
    await form.fill("データの説明 (日本語)", "データの説明")
    await form.fill("データの分野", "ライフサイエンス")
    await form.fill("データ種別", "集計データ")
    await form.fill("管理対象データの利活用・提供方針 (有償/無償)", "無償")
    await form.fill("管理対象データの利活用・提供方針 (ライセンス)", "CC0 1.0 Universal")
    await form.fill("管理対象データの利活用・提供方針 (引用方法等・日本語)", "引用方法の説明")
    await form.fill("Data utilization and provision policy (citation information, English)", "citation information")
    await form.fill("アクセス権", "公開")
    await form.scroll_to("データ管理機関 (日本語)")
    await form.fill("データ管理機関 (日本語)", "データ管理機関")
    await form.fill("データ管理者の種類", "個人")
    await form.fill_name("データ管理者 (日本語)", {'last': '情報', 'middle': 'データ', 'first': '太郎'})

await run_pw(_step)

## WEKOのURLを開き、「WorkFlow」をクリックする

WorkFlow画面が表示されること

In [ ]:
async def _step(page):
    await page.goto(weko_url)
    await expect(page.locator('//a[text() = "WorkFlow"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    
    global current_activities, weko_workflow_page
    await page.locator('//a[text() = "WorkFlow"]').click()
    await expect(page.locator('#myTabContent').locator('.table-responsive').locator('//th[text() = "Activity"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    current_activities = await page.locator('.activity-link').count()
    weko_workflow_page = page

await run_pw(_step, new_page=True)

## GakuNin RDMのページに戻り、「保存してJAIRO Cloudに登録」をクリックし、「プロジェクトA必須」を選択して「OK」をクリックする

ファイルビューに戻り、「JAIRO Cloud」配下に「サンプルデータ」アイテムが作成されること

ダイアログが表示されたら「OK」をクリックして閉じること。Directモードの場合は、「JAIRO Cloud」配下に「サンプルデータ」アイテムが作成されること

In [ ]:
async def _step(page):
    page = project_page
    await page.locator('//a[text()="保存してJAIRO Cloudに登録"]').click()
    await expect(page.locator('//h3[contains(text(), "ファイルの登録")]')).to_be_visible(timeout=transition_timeout)
    
    checkbox = page.locator('//div[@class="modal fade in"]//*[contains(text(), "プロジェクトA必須 (")]/..//input[@type = "checkbox"]')
    await checkbox.click()
    await expect(checkbox).to_be_checked(timeout=transition_timeout)
    
    await page.locator('//div[@class="modal fade in"]//a[text()="OK"]').click()
    await expect(get_result_dialog_locator(page)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    return page

await run_pw(_step)

## WEKOの画面に戻り、リロードする

workflowの場合、Activityが増えること

In [ ]:
async def _step(page):
    page = weko_workflow_page
    await page.reload()
    await expect(page.locator('#myTabContent').locator('.table-responsive').locator('//th[text() = "Activity"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    expected_activities = current_activities
    if oauth_client_type == 'workflow':
        expected_activities += 1
    await expect(page.locator('.activity-link')).to_have_count(expected_activities, timeout=transition_timeout)
    return page

await run_pw(_step)


## (Workflowの場合)一番上のActivityの「表示」をクリックし、「Approval」ボタンをクリック、その後「Access」ボタンをクリックする

アイテム詳細画面が表示されること

In [ ]:
async def _step(page):
    if oauth_client_type != 'workflow':
        return
    added_activity_link = page.locator('.activity-link').first
    await added_activity_link.click()

    await expect(page.locator('//button[text() = "Approval"]')).to_be_enabled(timeout=transition_timeout)
    await asyncio.sleep(5)
    await page.locator('//button[text() = "Approval"]').click()

    await expect(page.locator('//button[contains(text(), "Access")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    await page.locator('//button[contains(text(), "Access")]').click()

    await expect(page.locator('//a[contains(text(), "JSON")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#public_status_form')).to_contain_text('Public', timeout=transition_timeout)
    await asyncio.sleep(5)

await run_pw(_step)

## (Directの場合) 「JAIRO Cloud」配下の「サンプルデータ」アイテムをクリックする

アイテム詳細画面が表示されること

In [ ]:
async def _step(page):
    if oauth_client_type == 'workflow':
        await get_result_dialog_locator(project_page).locator('../../..//a[text()="OK"]').click()
        await asyncio.sleep(5)
        return
    page = project_page
    popup_future = page.context.wait_for_event('page')
    await get_result_dialog_locator(page).locator(f'../../..//a[contains(text(), "{weko_url}")]').click()
    popup = await popup_future
    await popup.wait_for_load_state()
    await get_result_dialog_locator(page).locator('../../..//a[text()="OK"]').click()
    await expect(popup.locator('//a[contains(text(), "JSON")]')).to_be_visible(timeout=transition_timeout)
    await expect(popup.locator('#public_status_form')).to_contain_text('Public', timeout=transition_timeout)
    await expect(popup.locator('//span[contains(@class, "filename")]//a[contains(normalize-space(.), "サンプル.png")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    return popup

await run_pw(_step)

## WEKOのJSON出力を確認する（必須項目のみ）

JSON出力に含まれるmetadataが以下の内容と一致すること

In [ ]:
async def _step(page):
    data_expected = """
      item_30002_title0:
        attribute_name: Title
        attribute_value_mlt:
        - subitem_title: サンプルデータ
          subitem_title_language: ja
      item_30002_access_rights4:
        attribute_name: Access Rights
        attribute_value_mlt:
        - subitem_access_right: open access
          subitem_access_right_uri: http://purl.org/coar/access_right/c_abf2
      item_30002_rights6:
        attribute_name: Rights
        attribute_value_mlt:
        - subitem_rights: citation information
          subitem_rights_language: en
        - subitem_rights: 引用方法の説明
          subitem_rights_language: ja
        - subitem_rights: free
          subitem_rights_language: en
        - subitem_rights: 無償
          subitem_rights_language: ja
        - subitem_rights: CC0 1.0 Universal
          subitem_rights_language: en
          subitem_rights_resource: https://creativecommons.org/publicdomain/zero/1.0/deed.en
        - subitem_rights: CC0 1.0 Universal
          subitem_rights_language: ja
          subitem_rights_resource: https://creativecommons.org/publicdomain/zero/1.0/deed.en
      item_30002_subject8:
        attribute_name: Subject
        attribute_value_mlt:
        - subitem_subject: Life Science
          subitem_subject_language: en
          subitem_subject_scheme: e-Rad_field
        - subitem_subject: ライフサイエンス
          subitem_subject_language: ja
          subitem_subject_scheme: e-Rad_field
      item_30002_description9:
        attribute_name: Description
        attribute_value_mlt:
        - subitem_description: データの説明
          subitem_description_language: ja
          subitem_description_type: Abstract
      item_30002_funding_reference21:
        attribute_name: Funding Reference
        attribute_value_mlt:
        - subitem_award_numbers:
            subitem_award_number: TEST123456
            subitem_award_number_type: JGN
          subitem_award_titles:
          - subitem_award_title: プロジェクトA必須
            subitem_award_title_language: ja
          subitem_funder_identifiers:
            subitem_funder_identifier_type: ROR
            subitem_funder_identifier: "https://ror.org/004rtk039"
          subitem_funder_names:
          - subitem_funder_name: Japan Agency for Medical Research and Development(AMED)
            subitem_funder_name_language: en
          - subitem_funder_name: 国立研究開発法人日本医療研究開発機構(AMED)
            subitem_funder_name_language: ja
      item_30002_resource_type13:
        attribute_name: Resource Type
        attribute_value_mlt:
        - resourcetype: aggregated data
          resourceuri: http://purl.org/coar/resource_type/ACF7-8YT9
      item_30002_contributor3:
        attribute_name: Contributor
        attribute_value_mlt:
        - contributorNames:
          - contributorName: "情報, データ 太郎"
            lang: ja
          contributorType: DataManager
          familyNames:
          - familyName: 情報
            familyNameLang: ja
          givenNames:
          - givenName: データ 太郎
            givenNameLang: ja
        - contributorNames:
          - contributorName: データ管理機関
            lang: ja
            nameType: Organizational
          contributorType: HostingInstitution

    """
    await page.locator('//a[contains(text(), "JSON")]').click()
    await expect(page.locator('//pre')).to_be_visible(timeout=transition_timeout)
    actual_metadata = json.loads(await page.locator('//pre').inner_text())['metadata']
    expected_metadata = yaml.safe_load(data_expected)
    for key, expected in expected_metadata.items():
        message = f'Missing key: {key} in actual metadata \nActual metadata: {json.dumps(actual_metadata, ensure_ascii=False, indent=2, sort_keys=True)}'
        assert key in actual_metadata, message
        actual = actual_metadata[key]
        assert key in actual_metadata, f'Missing key: {key}'
        actual_str = json.dumps(actual, ensure_ascii=False, indent=2, sort_keys=True)
        expected_str = json.dumps(expected, ensure_ascii=False, indent=2, sort_keys=True)
        diff = '\n'.join(
            difflib.unified_diff(
                expected_str.splitlines(),
                actual_str.splitlines(),
                fromfile='expected',
                tofile='actual',
                lineterm=''
            )
        )
        message = (
            f"Metadata mismatch for '{key}'.\n"
            f"{diff or 'No textual diff (objects differ structurally).'}"
        )
        assert actual == expected, message

await run_pw(_step)

## 「Delete」をクリックし、確認ダイアログで「OK」をクリックする

アイテムが削除されること

In [ ]:
async def _step(page):
    await page.go_back()
    await page.locator('//a[@id = "btn_delete"]').click()
    await expect(page.locator('//button[@id = "confirm_delete_button"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[@id = "confirm_delete_button"]').click()
    await expect(page.locator('//*[contains(text(), "This item has been deleted")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(transition_timeout / 1000 / 2)

await run_pw(_step)

## 「NII Storage」(または機関ストレージ)をクリックし、フォルダを作成する

「TESTMETADATA」フォルダが作成されること

In [ ]:
await close_latest_page()

async def _step(page):
    await goto_file_view(page)
    await expect(grdm.get_select_storage_title_locator(page, default_storage_label)).to_be_visible(timeout=transition_timeout)
    # Wait for loading modal to disappear
    await expect(page.locator('//div[contains(@class, "tb-modal-shade") and contains(@class, "loading-modal")]')).to_be_hidden(timeout=transition_timeout)
    await grdm.get_select_storage_title_locator(page, default_storage_label).click()
    await expect(page.locator('//*[text() = "新規フォルダ作成"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//*[text() = "新規フォルダ作成"]').click()
    await expect(page.locator('//input[@id = "createFolderInput"]')).to_be_visible(timeout=transition_timeout)
    folder_input = page.locator('//input[@id = "createFolderInput"]')
    await folder_input.fill('TESTMETADATA')
    await folder_input.press('Enter')
    await expect(grdm.get_select_folder_title_locator(page, 'TESTMETADATA')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「TESTMETADATA」フォルダに 2 つのファイルをアップロードする

「TESTMETADATA」フォルダ直下に `evidence_notes.docx` と `evidence_overview.txt` が表示されること


In [ ]:
async def _step(page):
    dropzone = grdm.get_select_folder_droppable_xpath('TESTMETADATA')
    for file_path in root_evidence_file_paths:
        await grdm.drop_file(page, dropzone, file_path)
        await grdm.wait_for_uploaded(page, os.path.basename(file_path))
    for file_path in root_evidence_file_paths:
        filename = os.path.basename(file_path)
        await expect(grdm.get_select_file_title_locator(page, filename)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「TESTMETADATA」フォルダのメタデータを編集する

メタデータが設定されること

In [ ]:
async def _step(page):
    await grdm.get_select_folder_title_locator(page, 'TESTMETADATA').click()
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//*[text() = "メタデータ編集"]').click()
    await expect(page.locator('//h3[contains(text(), "メタデータの編集")]')).to_be_visible(timeout=transition_timeout)
    
    schema_select = page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')
    await schema_select.select_option(label="研究成果（公的資金対応）のメタデータ登録")
    await expect(page.locator('//label[contains(text(), "データ No.")]')).to_be_visible(timeout=transition_timeout)
    
    form = FileMetadataForm(page)
    
    await form.fill("データの名称 (日本語)", "サンプルデータ")
    await form.fill("データの説明 (日本語)", "データの説明")
    await form.fill("データの分野", "ライフサイエンス")
    await form.fill("データ種別", "集計データ")
    await form.fill("管理対象データの利活用・提供方針 (有償/無償)", "無償")
    await form.fill("管理対象データの利活用・提供方針 (ライセンス)", "CC0 1.0 Universal")
    await form.fill("管理対象データの利活用・提供方針 (引用方法等・日本語)", "引用方法の説明")
    await form.fill("Data utilization and provision policy (citation information, English)", "citation information")
    await form.fill("アクセス権", "公開")
    await form.scroll_to("データ管理機関 (日本語)")
    await form.fill("データ管理機関 (日本語)", "データ管理機関")
    await form.fill("データ管理者の種類", "個人")
    await form.fill_name("データ管理者 (日本語)", {'last': '情報', 'middle': 'データ', 'first': '太郎'})
    
    await page.locator('//a[contains(text(), "保存")]').click()
    await expect(page.locator('//h3[contains(text(), "メタデータの編集")]')).to_be_hidden(timeout=transition_timeout)

await run_pw(_step)

## 「TESTMETADATA」を「JAIRO Cloud」にドラッグ＆ドロップする

フォルダがコピーされること

In [ ]:
import traceback

importlib.reload(scripts.grdm)
import scripts.grdm as grdm

async def _step(page):
    target_label = f'JAIRO Cloud: {weko_index_name}'
    source = grdm.get_select_folder_draggable_locator(page, 'TESTMETADATA')
    await expect(source).to_be_visible(timeout=transition_timeout)
    dest = page.locator(f'//*[contains(@class, "tb-expand-icon-holder")]//*[contains(@style, "/static/addons/weko/comicon.png")]/../../following-sibling::*[contains(@class, "title-text")]//*[text() = "{target_label}"]/../../../..')
    await expect(dest).to_be_visible(timeout=transition_timeout)
    await source.scroll_into_view_if_needed()
    await dest.scroll_into_view_if_needed()
    await source.click()
    await asyncio.sleep(15)
    await grdm.drag_and_drop(page, source, dest)
    try:
        # Check if registration dialog appears
        cancel_button = page.locator('//*[contains(text(), "JAIRO Cloudに登録しますか？")]/..//a[text() = "キャンセル"]')
        await expect(cancel_button).to_be_visible(timeout=transition_timeout)
        await cancel_button.click()
        await expect(cancel_button).not_to_be_visible(timeout=transition_timeout)
    except:
        # Pendingの場合は間をおいてリロード
        traceback.print_exc()
        await expect(page.locator('//*[contains(text(), "Your copy will still be completed")]')).to_be_visible(timeout=transition_timeout)
        await asyncio.sleep(transition_timeout)
        await page.reload()

await run_pw(_step)

## JAIRO Cloud配下でフォルダを確認し、「登録」をクリックする

ファイルの登録ダイアログが表示されること

In [ ]:
async def _step(page):
    target_label = f'JAIRO Cloud: {weko_index_name}'
    await grdm.get_select_storage_title_locator(page, target_label).click()
    await expect(page.locator('//*[text() = "新規フォルダ作成"]')).to_be_visible(timeout=transition_timeout)
    
    # Wait for TESTMETADATA to appear
    weko_row_locator = page.locator('.weko-row')
    await expect(weko_row_locator.locator(grdm.get_select_folder_title_xpath('TESTMETADATA'))).to_be_visible(timeout=transition_timeout)
    
    await weko_row_locator.locator(grdm.get_select_folder_title_xpath('TESTMETADATA')).click()
    await expect(page.locator('.weko-button-publish').locator('//*[text() = "登録"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('.weko-button-publish').locator('//*[text() = "登録"]').click()
    await expect(page.locator('//h3[contains(text(), "ファイルの登録")]')).to_be_visible(timeout=transition_timeout)
    global project_page
    project_page = page

await run_pw(_step)

## WEKOのURLを開き、「WorkFlow」をクリックする

WorkFlow画面が表示されること

In [ ]:
async def _step(page):
    await page.goto(weko_url)
    await expect(page.locator('//a[text() = "WorkFlow"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    
    global current_activities, weko_workflow_page
    await page.locator('//a[text() = "WorkFlow"]').click()
    await expect(page.locator('#myTabContent').locator('.table-responsive').locator('//th[text() = "Activity"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    current_activities = await page.locator('.activity-link').count()
    weko_workflow_page = page

await run_pw(_step, new_page=True)

## 「プロジェクトA必須」のチェックボックスをクリックし、「OK」をクリックする

「JAIRO Cloud」配下に「サンプルデータ」アイテムが作成されること

In [ ]:
async def _step(page):
    page = project_page
    checkbox = page.locator('//div[@class="modal fade in"]//*[contains(text(), "プロジェクトA必須")]/..//input[@type="checkbox"]')
    await checkbox.click()
    await expect(checkbox).to_be_checked(timeout=transition_timeout)
    
    await page.locator('//div[@class="modal fade in"]//a[text()="OK"]').click()
    await expect(get_result_dialog_locator(page)).to_be_visible(timeout=transition_timeout)
    await get_result_dialog_locator(page).locator('../../..//a[text()="OK"]').click()
    if oauth_client_type != 'workflow':
        await expect(page.locator('//*[contains(@class, "weko-row")]//*[text() = "サンプルデータ"]')).to_be_visible(timeout=transition_timeout)
    return page

await run_pw(_step)

## WEKOの画面に戻り、リロードする

workflowの場合、Activityが増えること

In [ ]:
async def _step(page):
    page = weko_workflow_page
    await page.reload()
    await expect(page.locator('#myTabContent').locator('.table-responsive').locator('//th[text() = "Activity"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    expected_activities = current_activities
    if oauth_client_type == 'workflow':
        expected_activities += 1
    await expect(page.locator('.activity-link')).to_have_count(expected_activities, timeout=transition_timeout)
    return page

await run_pw(_step)


## (Workflowの場合)一番上のActivityの「表示」をクリックし、「Approval」ボタンをクリック、その後「Access」ボタンをクリックする

アイテム詳細画面が表示されること

In [ ]:
async def _step(page):
    if oauth_client_type != 'workflow':
        return
    added_activity_link = page.locator('.activity-link').first
    await added_activity_link.click()

    await expect(page.locator('//button[text() = "Approval"]')).to_be_enabled(timeout=transition_timeout)
    await asyncio.sleep(5)
    await page.locator('//button[text() = "Approval"]').click()

    await expect(page.locator('//button[contains(text(), "Access")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    await page.locator('//button[contains(text(), "Access")]').click()

    await expect(page.locator('//a[contains(text(), "JSON")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#public_status_form')).to_contain_text('Public', timeout=transition_timeout)
    await asyncio.sleep(5)

await run_pw(_step)

## (Directの場合) 「JAIRO Cloud」配下の「サンプルデータ」アイテムをクリックする

アイテム詳細画面が表示されること

In [ ]:
async def _step(page):
    if oauth_client_type == 'workflow':
        return
    page = project_page
    popup_future = page.context.wait_for_event('page')
    await page.locator('//*[contains(@class, "weko-row")]//*[text() = "サンプルデータ"]').click()
    popup = await popup_future
    await popup.wait_for_load_state()
    await expect(popup.locator('//a[contains(text(), "JSON")]')).to_be_visible(timeout=transition_timeout)
    await expect(popup.locator('#public_status_form')).to_contain_text('Public', timeout=transition_timeout)
    for file_path in root_evidence_file_paths:
        filename = os.path.basename(file_path)
        await expect(popup.locator(f'//span[contains(@class, "filename")]//a[contains(normalize-space(.), "{filename}")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    return popup

await run_pw(_step)

## WEKOのJSON出力を確認する（フォルダ移動後）

JSON出力に含まれるmetadataが以下の内容と一致すること

In [ ]:
async def _step(page):
    data_expected = """
      item_30002_title0:
        attribute_name: Title
        attribute_value_mlt:
        - subitem_title: サンプルデータ
          subitem_title_language: ja
      item_30002_access_rights4:
        attribute_name: Access Rights
        attribute_value_mlt:
        - subitem_access_right: open access
          subitem_access_right_uri: http://purl.org/coar/access_right/c_abf2
      item_30002_rights6:
        attribute_name: Rights
        attribute_value_mlt:
        - subitem_rights: citation information
          subitem_rights_language: en
        - subitem_rights: 引用方法の説明
          subitem_rights_language: ja
        - subitem_rights: free
          subitem_rights_language: en
        - subitem_rights: 無償
          subitem_rights_language: ja
        - subitem_rights: CC0 1.0 Universal
          subitem_rights_language: en
          subitem_rights_resource: https://creativecommons.org/publicdomain/zero/1.0/deed.en
        - subitem_rights: CC0 1.0 Universal
          subitem_rights_language: ja
          subitem_rights_resource: https://creativecommons.org/publicdomain/zero/1.0/deed.en
      item_30002_subject8:
        attribute_name: Subject
        attribute_value_mlt:
        - subitem_subject: Life Science
          subitem_subject_language: en
          subitem_subject_scheme: e-Rad_field
        - subitem_subject: ライフサイエンス
          subitem_subject_language: ja
          subitem_subject_scheme: e-Rad_field
      item_30002_description9:
        attribute_name: Description
        attribute_value_mlt:
        - subitem_description: データの説明
          subitem_description_language: ja
          subitem_description_type: Abstract
      item_30002_funding_reference21:
        attribute_name: Funding Reference
        attribute_value_mlt:
        - subitem_award_numbers:
            subitem_award_number: TEST123456
            subitem_award_number_type: JGN
          subitem_award_titles:
          - subitem_award_title: プロジェクトA必須
            subitem_award_title_language: ja
          subitem_funder_identifiers:
            subitem_funder_identifier_type: ROR
            subitem_funder_identifier: "https://ror.org/004rtk039"
          subitem_funder_names:
          - subitem_funder_name: Japan Agency for Medical Research and Development(AMED)
            subitem_funder_name_language: en
          - subitem_funder_name: 国立研究開発法人日本医療研究開発機構(AMED)
            subitem_funder_name_language: ja
      item_30002_resource_type13:
        attribute_name: Resource Type
        attribute_value_mlt:
        - resourcetype: aggregated data
          resourceuri: http://purl.org/coar/resource_type/ACF7-8YT9
      item_30002_contributor3:
        attribute_name: Contributor
        attribute_value_mlt:
        - contributorNames:
          - contributorName: "情報, データ 太郎"
            lang: ja
          contributorType: DataManager
          familyNames:
          - familyName: 情報
            familyNameLang: ja
          givenNames:
          - givenName: データ 太郎
            givenNameLang: ja
        - contributorNames:
          - contributorName: データ管理機関
            lang: ja
            nameType: Organizational
          contributorType: HostingInstitution

    """
    await page.locator('//a[contains(text(), "JSON")]').click()
    await expect(page.locator('//pre')).to_be_visible(timeout=transition_timeout)
    actual_metadata = json.loads(await page.locator('//pre').inner_text())['metadata']
    expected_metadata = yaml.safe_load(data_expected)
    for key, expected in expected_metadata.items():
        message = f'Missing key: {key} in actual metadata \nActual metadata: {json.dumps(actual_metadata, ensure_ascii=False, indent=2, sort_keys=True)}'
        assert key in actual_metadata, message
        actual = actual_metadata[key]
        assert key in actual_metadata, f'Missing key: {key}'
        actual_str = json.dumps(actual, ensure_ascii=False, indent=2, sort_keys=True)
        expected_str = json.dumps(expected, ensure_ascii=False, indent=2, sort_keys=True)
        diff = '\n'.join(
            difflib.unified_diff(
                expected_str.splitlines(),
                actual_str.splitlines(),
                fromfile='expected',
                tofile='actual',
                lineterm=''
            )
        )
        message = (
            f"Metadata mismatch for '{key}'.\n"
            f"{diff or 'No textual diff (objects differ structurally).'}"
        )
        assert actual == expected, message

await run_pw(_step)

## 「Delete」をクリックし、確認ダイアログで「OK」をクリックする

アイテムが削除されること

In [ ]:
async def _step(page):
    await page.go_back()
    await page.locator('//a[@id = "btn_delete"]').click()
    await expect(page.locator('//button[@id = "confirm_delete_button"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[@id = "confirm_delete_button"]').click()
    await expect(page.locator('//*[contains(text(), "This item has been deleted")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(transition_timeout / 1000 / 2)

await run_pw(_step)


## WEKOのウィンドウを閉じる

GRDMのプロジェクトダッシュボードが表示されること


In [ ]:
await close_latest_page()

async def _step(page):
    await goto_file_view(page)
    await expect(grdm.get_select_storage_title_locator(page, default_storage_label)).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_folder_title_locator(page, 'TESTMETADATA')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「TESTMETADATA」をクリックする

「名前を変更」ボタンが表示されること

In [ ]:
async def _step(page):
    await grdm.get_select_folder_title_locator(page, 'TESTMETADATA').click()
    await expect(page.locator('//*[text() = "名前を変更"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## ファイル一覧の上部メニューから「名前を変更」をクリックする

In [ ]:
async def _step(page):
    await page.locator('//i[contains(@class, "fa-pencil")]/../*[text() = "名前を変更"]').click()

    await expect(page.locator('#renameInput')).to_be_editable()

await run_pw(_step)

## テキスト入力フォームの内容を「根拠データ」に編集してEnterキーを押下する

In [ ]:
async def _step(page):
    rename_input = page.locator('#renameInput')
    new_filename = "根拠データ"
    await rename_input.fill(new_filename)
    await rename_input.press('Enter')
    await expect(grdm.get_select_folder_title_locator(page, "TESTMETADATA")).to_have_count(0, timeout=transition_timeout)
    await expect(grdm.get_select_folder_title_locator(page, new_filename)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「根拠データ」をクリックする

「メタデータ編集」ボタンが表示されること

In [ ]:
async def _step(page):
    await grdm.get_select_folder_title_locator(page, '根拠データ').click()
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「メタデータ編集」をクリックする

「メタデータ編集」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()
    await expect(page.locator('//h3[contains(text(), "ファイルメタデータの編集")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 全ての入力欄に正常系の値を入力する

正常系の値は以下の通り:
- データ No.: 123
- データの名称 (日本語): サンプルデータ
- Title (English): sample data
- 掲載日・掲載更新日: 2030-01-01
- データの説明 (日本語): データの説明
- Description (English): data description
- データの分野: ライフサイエンス
- データ種別: 集計データ
- 概略データ量: 1KB
- 管理対象データの利活用・提供方針 (有償/無償): 無償
- 管理対象データの利活用・提供方針 (ライセンス): CC0 1.0 Universal
- 管理対象データの利活用・提供方針 (引用方法等・日本語): 引用方法の説明
- Data utilization and provision policy (citation information, English): citation information
- アクセス権: 公開
- 公開予定日: 2030-01-01
- リポジトリ情報 (日本語): リポジトリの情報
- Repository information (English): repository information
- リポジトリURL・DOIリンク: https://example.com/example
- データ作成者: [{e-Rad: '123', 日本語: '情報データ太郎', English: 'Taro Data Joho'}, {e-Rad: '456', 日本語: 'あああ', English: 'aaa'}]
- データ管理機関 (日本語): データ管理機関
- Hosting institution (English): Hosting institution
- データ管理機関コード: code
- データ管理者の種類: 個人
- データ管理者の e-Rad 研究者番号: 123
- データ管理者 (日本語): 情報データ太郎
- Data manager (English): Taro Data Joho
- データ管理者の所属組織名 (日本語): データ管理者の所属組織名
- Contact organization of data manager (English): Contact organization of data manager
- データ管理者の所属機関の連絡先住所 (日本語): 愛知県 名古屋市
- Contact address of data manager (English): Aichi, Nagoya
- データ管理者の所属機関の連絡先電話番号: 123-456-789
- データ管理者の所属機関の連絡先メールアドレス: sample@example.com
- 備考 (日本語): 備考123
- Remarks (English): Remarks123

各フィールドに入力した値が表示されること

In [ ]:
importlib.reload(scripts.metadata_v2025)
from scripts.metadata_v2025 import ProjectMetadataForm, FileMetadataForm

async def _step(page):
    form = FileMetadataForm(page)
    
    await form.fill("データ No.", "123")
    await form.fill("データの名称 (日本語)", "サンプルデータ")
    await form.fill("Data title (English)", "sample data")
    await form.fill("掲載日・掲載更新日", "2030-01-01")
    await form.fill("データの説明 (日本語)", "データの説明")
    await form.fill("Description (English)", "data description")
    await form.fill("データの分野", "ライフサイエンス")
    await form.fill("データ種別", "集計データ")
    await form.fill("概略データ量", "1KB")
    await form.fill("管理対象データの利活用・提供方針 (有償/無償)", "無償")
    await form.fill("管理対象データの利活用・提供方針 (ライセンス)", "CC0 1.0 Universal")
    await form.fill("管理対象データの利活用・提供方針 (引用方法等・日本語)", "引用方法の説明")
    await form.fill("Data utilization and provision policy (citation information, English)", "citation information")
    await form.fill("アクセス権", "公開")
    await form.fill("公開予定日", "2030-01-01")
    await form.fill("リポジトリ情報 (日本語)", "リポジトリの情報")
    await form.fill("Repository information (English)", "repository information")
    await form.fill("リポジトリURL・DOIリンク", "https://example.com/example")
    
    # Table: データ作成者
    await form.fill_author({
        'number': '123',
        'name_ja': {'last': '情報', 'middle': 'データ', 'first': '太郎'},
        'name_en': {'last': 'Joho', 'middle': 'Data', 'first': 'Taro'},
        'affiliation_ja': '',
        'affiliation_en': '',
    }, table_label='データ作成者', add_row=False)
    await form.fill_author({
        'number': '456',
        'name_ja': {'last': 'あ', 'middle': 'あ', 'first': 'あ'},
        'name_en': {'last': 'a', 'middle': 'a', 'first': 'a'},
        'affiliation_ja': '',
        'affiliation_en': '',
    }, table_label='データ作成者')
    
    await form.scroll_to("データ管理機関 (日本語)")
    await form.fill("データ管理機関 (日本語)", "データ管理機関")
    await form.fill("Hosting institution (English)", "Hosting institution")
    await form.fill("データ管理機関コード", "code")
    await form.fill("データ管理者の種類", "個人")
    await form.fill("データ管理者の e-Rad 研究者番号", "123")
    await form.fill_name("データ管理者 (日本語)", {'last': '情報', 'middle': 'データ', 'first': '太郎'})
    await form.fill_name("Data manager (English)", {"last": "Joho", "middle": "Data", "first": "Taro"})
    await form.fill("データ管理者の所属組織名 (日本語)", "データ管理者の所属組織名")
    await form.fill("Contact organization of data manager (English)", "Contact organization of data manager")
    await form.fill("データ管理者の所属機関の連絡先住所 (日本語)", "愛知県 名古屋市")
    await form.fill("Contact address of data manager (English)", "Aichi, Nagoya")
    await form.fill("データ管理者の所属機関の連絡先電話番号", "123-456-789")
    await form.fill("データ管理者の所属機関の連絡先メールアドレス", "sample@example.com")
    await form.fill("備考 (日本語)", "備考123")
    await form.fill("Remarks (English)", "Remarks123")
    
    # Verify some key values
    await form.scroll_to("データ No.")
    assert await form.get("データ No.") == "123"
    assert await form.get("データの名称 (日本語)") == "サンプルデータ"

await run_pw(_step)

## 「保存」をクリックする

ファイルメタデータの編集ダイアログが非表示になること

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    
    await page.locator('//a[text()="保存"]').click()
    await expect(form.get_locator("データ No.")).not_to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(3)

await run_pw(_step)

## 「根拠データ」をクリックする

「メタデータ登録」ボタンが表示されること

In [ ]:
async def _step(page):
    await grdm.get_select_folder_title_locator(page, '根拠データ').click()
    await expect(page.locator('//*[text() = "メタデータ登録"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

##  「メタデータ登録」をクリックする

「ファイルメタデータ登録先の選択」ダイアログが表示され、「プロジェクトA必須」とチェックボックスが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ登録"]').click()
    await expect(page.locator('//input[starts-with(@id, "draft-")]')).to_have_count(2, timeout=transition_timeout)

await run_pw(_step)

##  「プロジェクトA」をクリックする

チェックボックスがチェック状態になること

In [ ]:
async def _step(page):
    target = page.locator('//*[text() = "プロジェクトA"]')
    checkbox = target.locator('..//input[@type = "checkbox"]')
    await checkbox.click()
    await expect(checkbox).to_be_checked(timeout=transition_timeout)

await run_pw(_step)


##  「選択」をクリックする

「プロジェクトA」に「開く」リンクが現れること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "選択"]').click()
    await expect(page.locator('//*[text() = "開く"]')).to_be_enabled(timeout=transition_timeout)
    await asyncio.sleep(3)

await run_pw(_step)

## 「開く」をクリックする

ワークフローメタデータ編集画面が表示され、登録データ一覧に「根拠データ」がチェック状態で表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "開く"]').click()
    await expect(page.locator('//*[@data-test-goto-review]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//p[contains(., "根拠データ")]/../..//*[contains(@class, "fa-check-square")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「内容確認」をクリックする

左側ペイン「登録データ」が緑色かつチェックマークが表示されること

In [ ]:
import re

async def _step(page):
    await page.locator('//*[@data-test-goto-review]').click()
    
    await expect(page.locator('//span[@data-test-label and text() = "登録データ"]/../preceding-sibling::i')).to_have_class(re.compile(r'.*fa-check-circle-o.*'), timeout=transition_timeout)

await run_pw(_step)

## 「エクスポート」をクリックする

エクスポート先選択画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[@data-test-registration-card-export]').click()
    
    await expect(page.locator('//select[@id = "registration-report-format-selection"]')).to_be_enabled(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「リポジトリ (JAIRO Cloud)に登録 - {weko_index_name}」を選択する

インデックス名を含むWEKO送信先のオプションが表示されることを確認する

In [ ]:
async def _step(page):
    target_option = f'リポジトリ (JAIRO Cloud)に登録 - {weko_index_name}'
    select = page.locator('//*[@id = "registration-report-format-selection"]')
    await select.select_option(label=target_option)
    global project_page
    project_page = page

await run_pw(_step)

## WEKOのURLを開き、「WorkFlow」をクリックする

WorkFlow画面が表示されること

In [ ]:
async def _step(page):
    await page.goto(weko_url)
    await expect(page.locator('//a[text() = "WorkFlow"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    
    global current_activities, weko_workflow_page
    await page.locator('//a[text() = "WorkFlow"]').click()
    await expect(page.locator('#myTabContent').locator('.table-responsive').locator('//th[text() = "Activity"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    current_activities = await page.locator('.activity-link').count()
    weko_workflow_page = page

await run_pw(_step, new_page=True)

## GakuNin RDM画面に戻り、「エクスポート」をクリックしてWEKO送信を開始する

WEKO送信ボタンを押して処理が始まること

In [ ]:
async def _step(page):
    page = project_page
    submit_button = page.locator('//*[@data-test-registration-report-submit]')
    await expect(submit_button).to_be_enabled(timeout=transition_timeout)
    await submit_button.click()
    await expect(submit_button).to_contain_text('エクスポート中', timeout=transition_timeout)
    await expect(page.locator('//*[@data-analytics-scope="RegistrationReport result modal"]')).to_be_visible(timeout=transition_timeout * 2)
    await asyncio.sleep(5)
    return page

await run_pw(_step)

## WEKOの画面に戻り、リロードする

workflowの場合、Activityが増えること

In [ ]:
async def _step(page):
    page = weko_workflow_page
    await page.reload()
    await expect(page.locator('#myTabContent').locator('.table-responsive').locator('//th[text() = "Activity"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    expected_activities = current_activities
    if oauth_client_type == 'workflow':
        expected_activities += 1
    await expect(page.locator('.activity-link')).to_have_count(expected_activities, timeout=transition_timeout)
    return page

await run_pw(_step)


## (Workflowの場合)一番上のActivityの「表示」をクリックし、「Approval」ボタンをクリック、その後「Access」ボタンをクリックする

アイテム詳細画面が表示されること

In [ ]:
async def _step(page):
    if oauth_client_type != 'workflow':
        return
    added_activity_link = page.locator('.activity-link').first
    await added_activity_link.click()

    await expect(page.locator('//button[text() = "Approval"]')).to_be_enabled(timeout=transition_timeout)
    await asyncio.sleep(5)
    await page.locator('//button[text() = "Approval"]').click()

    await expect(page.locator('//button[contains(text(), "Access")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(5)
    await page.locator('//button[contains(text(), "Access")]').click()

    await expect(page.locator('//a[contains(text(), "JSON")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#public_status_form')).to_contain_text('Public', timeout=transition_timeout)
    await asyncio.sleep(5)

await run_pw(_step)

## (Directの場合)GakuNin RDM画面でWEKOのアイテムリンクをクリックする

アイテム詳細画面が表示されること

In [ ]:
async def _step(page):
    if oauth_client_type == 'workflow':
        return
    page = project_page
    popup_future = page.context.wait_for_event('page')
    await page.locator('//a[@data-test-registration-report-result-link]').click()
    popup = await popup_future
    await popup.wait_for_load_state()
    await expect(popup.locator('//a[contains(text(), "JSON")]')).to_be_visible(timeout=transition_timeout)
    await expect(popup.locator('#public_status_form')).to_contain_text('Public', timeout=transition_timeout)
    await asyncio.sleep(5)
    return popup

await run_pw(_step)

## WEKOのJSON出力を確認する（全項目入力）

JSON出力に含まれるmetadataが以下の内容と一致すること

In [ ]:
import difflib

async def _step(page):
    data_expected = """
      item_30002_title0:
        attribute_name: Title
        attribute_value_mlt:
        - subitem_title: sample data
          subitem_title_language: en
        - subitem_title: サンプルデータ
          subitem_title_language: ja
      item_30002_creator2:
        attribute_name: Creator
        attribute_type: creator
        attribute_value_mlt:
        - creatorNames:
          - creatorName: "Joho, Data Taro"
            creatorNameLang: en
          - creatorName: "情報, データ 太郎"
            creatorNameLang: ja
          familyNames:
          - familyName: Joho
            familyNameLang: en
          - familyName: 情報
            familyNameLang: ja
          givenNames:
          - givenName: Data Taro
            givenNameLang: en
          - givenName: データ 太郎
            givenNameLang: ja
          nameIdentifiers:
          - nameIdentifier: '123'
            nameIdentifierScheme: e-Rad_Researcher
        - creatorNames:
          - creatorName: "a, a a"
            creatorNameLang: en
          - creatorName: "あ, あ あ"
            creatorNameLang: ja
          familyNames:
          - familyName: a
            familyNameLang: en
          - familyName: あ
            familyNameLang: ja
          givenNames:
          - givenName: a a
            givenNameLang: en
          - givenName: あ あ
            givenNameLang: ja
          nameIdentifiers:
          - nameIdentifier: '456'
            nameIdentifierScheme: e-Rad_Researcher
      item_30002_access_rights4:
        attribute_name: Access Rights
        attribute_value_mlt:
        - subitem_access_right: open access
          subitem_access_right_uri: http://purl.org/coar/access_right/c_abf2
      item_30002_rights6:
        attribute_name: Rights
        attribute_value_mlt:
        - subitem_rights: citation information
          subitem_rights_language: en
        - subitem_rights: 引用方法の説明
          subitem_rights_language: ja
        - subitem_rights: free
          subitem_rights_language: en
        - subitem_rights: 無償
          subitem_rights_language: ja
        - subitem_rights: CC0 1.0 Universal
          subitem_rights_language: en
          subitem_rights_resource: https://creativecommons.org/publicdomain/zero/1.0/deed.en
        - subitem_rights: CC0 1.0 Universal
          subitem_rights_language: ja
          subitem_rights_resource: https://creativecommons.org/publicdomain/zero/1.0/deed.en
      item_30002_subject8:
        attribute_name: Subject
        attribute_value_mlt:
        - subitem_subject: Life Science
          subitem_subject_language: en
          subitem_subject_scheme: e-Rad_field
        - subitem_subject: ライフサイエンス
          subitem_subject_language: ja
          subitem_subject_scheme: e-Rad_field
      item_30002_description9:
        attribute_name: Description
        attribute_value_mlt:
        - subitem_description: data description
          subitem_description_language: en
          subitem_description_type: Abstract
        - subitem_description: データの説明
          subitem_description_language: ja
          subitem_description_type: Abstract
      # TODO: item_30002_date11 is not present in actual metadata - needs investigation
      item_30002_funding_reference21:
        attribute_name: Funding Reference
        attribute_value_mlt:
        - subitem_award_numbers:
            subitem_award_number: TEST123456
            subitem_award_number_type: JGN
          subitem_award_titles:
          - subitem_award_title: ProjectA
            subitem_award_title_language: en
          - subitem_award_title: プロジェクトA
            subitem_award_title_language: ja
          subitem_funder_identifiers:
            subitem_funder_identifier_type: ROR
            subitem_funder_identifier: "https://ror.org/004rtk039"
          subitem_funder_names:
          - subitem_funder_name: Japan Agency for Medical Research and Development(AMED)
            subitem_funder_name_language: en
          - subitem_funder_name: 国立研究開発法人日本医療研究開発機構(AMED)
            subitem_funder_name_language: ja
          subitem_funding_stream_identifiers:
            subitem_funding_stream_identifier: abc12
            subitem_funding_stream_identifier_type: JGN_fundingStream
          subitem_funding_streams:
          - subitem_funding_stream: ProgramA
            subitem_funding_stream_language: en
          - subitem_funding_stream: プログラムA
            subitem_funding_stream_language: ja
      item_30002_resource_type13:
        attribute_name: Resource Type
        attribute_value_mlt:
        - resourcetype: aggregated data
          resourceuri: http://purl.org/coar/resource_type/ACF7-8YT9
      item_30002_contributor3:
        attribute_name: Contributor
        attribute_value_mlt:
        - contributorNames:
          - contributorName: 'Contact organization of data manager Aichi, Nagoya TEL:
              123-456-789 E-Mail: sample@example.com'
            lang: en
            nameType: Organizational
          - contributorName: 'データ管理者の所属組織名 愛知県 名古屋市 TEL: 123-456-789 E-Mail: sample@example.com'
            lang: ja
            nameType: Organizational
          contributorType: ContactPerson
        - contributorNames:
          - contributorName: "Joho, Data Taro"
            lang: en
          - contributorName: "情報, データ 太郎"
            lang: ja
          contributorType: DataManager
          familyNames:
          - familyName: Joho
            familyNameLang: en
          - familyName: 情報
            familyNameLang: ja
          givenNames:
          - givenName: Data Taro
            givenNameLang: en
          - givenName: データ 太郎
            givenNameLang: ja
          nameIdentifiers:
          - nameIdentifierScheme: e-Rad_Researcher
            nameIdentifier: '123'
          contributorAffiliations:
          - contributorAffiliationNames:
            - contributorAffiliationName: Contact organization of data manager
              contributorAffiliationNameLang: en
            - contributorAffiliationName: データ管理者の所属組織名
              contributorAffiliationNameLang: ja
        - contributorNames:
          - contributorName: Hosting institution
            lang: en
            nameType: Organizational
          - contributorName: データ管理機関
            lang: ja
            nameType: Organizational
          contributorType: HostingInstitution
          nameIdentifiers:
          - nameIdentifierScheme: ROR
            nameIdentifier: code

    """
    await page.locator('//a[contains(text(), "JSON")]').click()
    await expect(page.locator('//pre')).to_be_visible(timeout=transition_timeout)
    actual_metadata = json.loads(await page.locator('//pre').inner_text())['metadata']
    expected_metadata = yaml.safe_load(data_expected)
    for key, expected in expected_metadata.items():
        message = f'Missing key: {key} in actual metadata \nActual metadata: {json.dumps(actual_metadata, ensure_ascii=False, indent=2, sort_keys=True)}'
        assert key in actual_metadata, message
        actual = actual_metadata[key]
        assert key in actual_metadata, f'Missing key: {key}'
        actual_str = json.dumps(actual, ensure_ascii=False, indent=2, sort_keys=True)
        expected_str = json.dumps(expected, ensure_ascii=False, indent=2, sort_keys=True)
        diff = '\n'.join(
            difflib.unified_diff(
                expected_str.splitlines(),
                actual_str.splitlines(),
                fromfile='expected',
                tofile='actual',
                lineterm=''
            )
        )
        message = (
            f"Metadata mismatch for '{key}'.\n"
            f"{diff or 'No textual diff (objects differ structurally).'}"
        )
        assert actual == expected, message

await run_pw(_step)

## (Directの場合) WEKOタブを閉じ、Registration Reportページをリロードする

「登録済」のステータスが表示されること

In [ ]:
async def _step(page):
    if oauth_client_type != 'direct':
        return
    await close_latest_page()
    page = project_page
    await page.reload()
    await expect(page.locator('//strong[text()="登録済"]')).to_be_visible(timeout=transition_timeout)
    return page

await run_pw(_step)

## (Directの場合) ファイルページに戻り、「根拠データ」の「データの説明」を変更する

「データの説明 (日本語)」を「データの説明（更新済み）」に変更し、保存すること

In [ ]:
async def _step(page):
    if oauth_client_type != 'direct':
        return
    global registration_report_url
    registration_report_url = project_page.url
    page = project_page
    await goto_file_view(page)
    await expect(grdm.get_select_folder_title_locator(page, '根拠データ')).to_be_visible(timeout=transition_timeout)
    await grdm.get_select_folder_title_locator(page, '根拠データ').click()
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)
    await page.locator('//*[text() = "メタデータ編集"]').click()
    await expect(page.locator('//h3[contains(text(), "ファイルメタデータの編集")]')).to_be_visible(timeout=transition_timeout)
    form = FileMetadataForm(page)
    await form.fill("データの説明 (日本語)", "データの説明（更新済み）")
    await page.locator('//a[text()="保存"]').click()
    await expect(form.get_locator("データ No.")).not_to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(3)

await run_pw(_step)

## (Directの場合) Registration Reportページに戻る

内容確認ページが表示されること

In [ ]:
async def _step(page):
    if oauth_client_type != 'direct':
        return
    page = project_page
    await page.goto(registration_report_url)
    await expect(page.locator('//button[@data-test-registration-card-export]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (Directの場合) 「エクスポート」をクリックし、WEKO送信先を選択する

エクスポート先選択画面が表示され、WEKO送信先を選択できること

In [ ]:
async def _step(page):
    if oauth_client_type != 'direct':
        return
    page = project_page
    await page.locator('//button[@data-test-registration-card-export]').click()
    await expect(page.locator('//select[@id = "registration-report-format-selection"]')).to_be_enabled(timeout=transition_timeout)
    time.sleep(1)
    target_option = f'リポジトリ (JAIRO Cloud)に登録 - {weko_index_name}'
    select = page.locator('//*[@id = "registration-report-format-selection"]')
    await select.select_option(label=target_option)

await run_pw(_step)

## (Directの場合) 「エクスポート」をクリックしてWEKO送信を開始する

WEKO送信ボタンを押して処理が始まること

In [ ]:
async def _step(page):
    if oauth_client_type != 'direct':
        return
    page = project_page
    submit_button = page.locator('//*[@data-test-registration-report-submit]')
    await expect(submit_button).to_be_enabled(timeout=transition_timeout)
    await submit_button.click()
    await expect(submit_button).to_contain_text('エクスポート中', timeout=transition_timeout)
    await expect(page.locator('//*[@data-analytics-scope="RegistrationReport result modal"]')).to_be_visible(timeout=transition_timeout * 2)
    await asyncio.sleep(5)
    return page

await run_pw(_step)

## (Directの場合) GakuNin RDM画面でWEKOのアイテムリンクをクリックする

アイテム詳細画面が表示されること

In [ ]:
async def _step(page):
    if oauth_client_type != 'direct':
        return
    page = project_page
    popup_future = page.context.wait_for_event('page')
    await page.locator('//a[@data-test-registration-report-result-link]').click()
    popup = await popup_future
    await popup.wait_for_load_state()
    await expect(popup.locator('//a[contains(text(), "JSON")]')).to_be_visible(timeout=transition_timeout)
    await expect(popup.locator('#public_status_form')).to_contain_text('Public', timeout=transition_timeout)
    await asyncio.sleep(5)
    return popup

await run_pw(_step)

## (Directの場合) WEKOのJSON出力を確認する（アイテム更新後）

「データの説明 (日本語)」が「データの説明（更新済み）」に変更されていること

In [ ]:
async def _step(page):
    if oauth_client_type != 'direct':
        return
    await page.locator('//a[contains(text(), "JSON")]').click()
    await expect(page.locator('//pre')).to_be_visible(timeout=transition_timeout)
    actual_metadata = json.loads(await page.locator('//pre').inner_text())['metadata']
    description = actual_metadata['item_30002_description9']
    ja_descriptions = [d for d in description['attribute_value_mlt'] if d.get('subitem_description_language') == 'ja']
    assert len(ja_descriptions) == 1, f'Expected 1 Japanese description, got {len(ja_descriptions)}'
    assert ja_descriptions[0]['subitem_description'] == 'データの説明（更新済み）', f'Expected updated description, got {ja_descriptions[0]["subitem_description"]}'

await run_pw(_step)

## WEKOで「Delete」をクリックし、アイテムを削除する

アイテムが削除されること

In [ ]:
async def _step(page):
    await page.go_back()
    await page.locator('//a[@id = "btn_delete"]').click()
    await expect(page.locator('//button[@id = "confirm_delete_button"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[@id = "confirm_delete_button"]').click()
    await expect(page.locator('//*[contains(text(), "This item has been deleted")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(transition_timeout / 1000 / 2)

await run_pw(_step)

In [ ]:
await close_latest_page()

後始末

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)

In [ ]:
!rm -fr {work_dir}